# BertDiffused — Baseline Comparison on LM1B Test Set

Runs the **pretrained baselines** (GPT-2 117M, GPT-2 Medium 345M) together with the
**trained BertMoEDiffusion checkpoint** on the LM1B `test` split and produces
comparison tables and plots — strictly evaluation, no training.

| Model | Family | Params | Source |
|---|---|---|---|
| **BertDiffused (ours)** | Masked Diffusion + MoE | 132M active / 275M total | `best_model/best_model.pt` |
| GPT-2 (117M) | Autoregressive | 117M | `openai-community/gpt2` |
| GPT-2 Medium (345M) | Autoregressive | 345M | `openai-community/gpt2-medium` |

Benchmarks:
1. **Text infilling** — BLEU-4, Gen-PPL
2. **Keyword-constrained generation** — KW-Sat %, MAUVE, Gen-PPL
3. **Unconditional generation** — Gen-PPL
4. **Test-set likelihood** — diffusion NELBO (nats/token) vs GPT-2 CE (nats/token), both reported as bits/token for a single-axis comparison

> Running the full 500-example evaluation on CPU is slow. Set `NUM_SAMPLES` small
> (e.g. 16) for a smoke test, then bump it up once you're on a GPU host.

## 0. Setup

Detects GPU, adds the repo root to `sys.path`, loads the config. Works on Colab
(mounts Drive + clones the repo) and locally.

In [ ]:
import os, sys, json, time, math
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

# ── Colab bootstrap (no-op locally) ──────────────────────────────────────
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    if not os.path.exists('/content/BertDiffused'):
        !git clone https://github.com/stephenlee/BertDiffused.git /content/BertDiffused
    os.chdir('/content/BertDiffused')
    !pip install -q -r requirements.txt

# ── Repo root on sys.path ────────────────────────────────────────────────
REPO_ROOT = Path('.').resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'configs' / 'config.yaml').exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Repo root: {REPO_ROOT}")
print(f"Device:    {device}")
if device.type == 'cuda':
    p = torch.cuda.get_device_properties(0)
    print(f"GPU:       {p.name} ({p.total_memory / 1e9:.1f} GB)")

In [ ]:
import yaml

with open('configs/config.yaml') as f:
    cfg = yaml.safe_load(f)

# ─── Knobs you can tune ─────────────────────────────────────────────────────
# Defaults target a CPU smoke-run. Bump everything up on a GPU host.
NUM_SAMPLES    = 8         # test-set examples per task (500 in the paper run)
NUM_STEPS      = 20        # diffusion reverse steps for Task 1 & 2
STEPS_SWEEP    = [10, 25, 50]    # Task 3 unconditional PPL sweep
SEQ_LEN        = 64        # infilling sequence length
KW_SEQ_LEN     = 48        # constrained-gen sequence length
BPD_BATCHES    = 2         # test-set likelihood batches
BPD_BATCH_SIZE = 4         # examples per batch
BPD_NUM_T      = 3         # Monte-Carlo t samples for diffusion NELBO
CKPT_PATH      = 'best_model/best_model.pt'
OUTPUT_DIR     = Path('results/baselines')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"NUM_SAMPLES = {NUM_SAMPLES}")
print(f"NUM_STEPS   = {NUM_STEPS}")
print(f"STEPS_SWEEP = {STEPS_SWEEP}")
print(f"CKPT_PATH   = {CKPT_PATH}  ({'found' if Path(CKPT_PATH).exists() else 'MISSING'})")
print(f"OUTPUT_DIR  = {OUTPUT_DIR}")

## 0b. Test corpus

Pulls a slice of the LM1B test split. Falls back to `wikitext-2-raw` (also on
HF hub) if `lm1b`'s dataset script fails to load, and to a hard-coded list of
news-wire sentences if both fail. This keeps the notebook runnable in any
environment — the rest of the notebook doesn't care how `TEST_TEXTS` was built.

In [ ]:
FALLBACK_SENTENCES = [
    "The central bank announced a sharp cut in interest rates on Tuesday morning in response to the deepening economic slowdown.",
    "Researchers at the university published a new study showing that a common household chemical can trigger allergic reactions.",
    "The team won its tenth consecutive game after a late three-point shot gave them a narrow one-point lead over their rivals.",
    "Flight delays at the regional airport lasted throughout the afternoon because of persistent heavy fog and low visibility.",
    "The governor signed the education bill on Friday, calling it the most significant reform package in more than two decades.",
    "A mountain rescue team recovered two climbers who had been trapped by a sudden storm high above the tree line.",
    "Stock markets rallied on news that a trade deal between the two largest economies in the world was close to being signed.",
    "The city council voted unanimously to expand the network of bicycle lanes along the downtown waterfront corridor.",
    "A new species of deep sea fish was discovered by an oceanographic expedition exploring a previously uncharted trench.",
    "The museum opened a major retrospective on Tuesday showcasing five decades of work by the pioneering photographer.",
    "Police said they are searching for a suspect who fled the scene of a robbery at a jewelry store in the historic district.",
    "The factory plans to hire more than two hundred new workers next quarter to meet rising demand for electric vehicle batteries.",
    "The director of the hospital confirmed that all patients had been evacuated safely after a small fire broke out in the east wing.",
    "A software engineer in the city developed an app that helps commuters find available parking spaces in real time.",
    "The author's latest novel has been translated into twelve languages and sold more than a million copies across Europe.",
    "Officials warned residents of the coastal town to prepare for heavy rainfall and possible flooding over the weekend.",
    "The court ruled that the company must pay damages to workers who had been exposed to harmful chemicals for many years.",
    "A solar panel factory in the suburb began shipping its first commercial product to customers in more than fifteen countries.",
    "Fans cheered as the orchestra finished the symphony with a dramatic crescendo that left the hall in stunned silence.",
    "The scientists concluded that the ancient pottery fragments were far older than any other artifacts found at the site.",
    "The farmer said the unusually dry summer had damaged nearly half of his corn crop and most of his soybean fields.",
    "A local charity collected more than ten thousand coats over the weekend to distribute to homeless families across the region.",
    "The prime minister pledged new infrastructure spending in a speech to parliament that focused heavily on rural development.",
    "Engineers working on the bridge project have completed the first of three main suspension cables ahead of schedule.",
    "The festival attracted record crowds this year and contributed significantly to the city's tourism and hospitality revenue.",
    "The professor argued that modern economic theory has failed to account for the long-term effects of environmental damage.",
    "A train derailed on the outskirts of town early this morning, though no injuries have been reported to local authorities.",
    "Voters in the coastal region turned out in unusually high numbers for what analysts are calling a pivotal local election.",
    "The restaurant chain plans to open fifty new locations across the midwest over the next two years, its chief executive said.",
    "A landmark trade agreement was signed late Friday between three neighboring countries aimed at boosting cross-border commerce.",
    "The pilot managed to land the small aircraft safely on the highway after the engine failed during a routine training flight.",
    "The children laughed and played in the park for most of the afternoon while their parents enjoyed a quiet picnic lunch.",
    "The technology conference opened Monday with a keynote address focused on the ethical implications of powerful generative models.",
    "A soccer team from the small island nation stunned its larger opponent with a dramatic last-minute goal in the quarterfinals.",
    "Economists say the new inflation figures suggest that price pressures may finally be easing after nearly three years of rapid rises.",
    "The explorer described the vast frozen landscape as both hauntingly beautiful and unforgiving, with temperatures well below zero.",
    "The hospital expanded its neonatal intensive care unit this spring, doubling the number of beds available for premature infants.",
    "The music festival featured more than eighty bands across six stages spread throughout the riverside park over three long days.",
    "A historian uncovered letters from the nineteenth century that shed new light on the political negotiations leading up to the war.",
    "The CEO announced at the annual meeting that the company would invest heavily in renewable energy over the coming decade.",
    "A volunteer crew spent the entire weekend cleaning the beach after a severe storm washed debris onto the usually pristine sand.",
    "The archaeological team reported finding the well-preserved remains of a temple dating back more than two thousand years.",
    "The mayor unveiled plans for a new public library in the historic district, to be funded largely by private donations.",
    "The startup's flagship product received glowing reviews from technology columnists, who praised its intuitive interface and clean design.",
    "The film director thanked the cast and crew during an emotional acceptance speech at the annual awards ceremony.",
    "Schools across the county are participating in a new reading program designed to improve literacy rates among young children.",
    "The author's latest book explores the impact of social media on political discourse during a turbulent election cycle.",
    "The architect explained that the new stadium's roof was specifically engineered to withstand extreme wind and snow loads.",
    "Commuters faced long delays on the subway this morning after a signaling fault disrupted service on several major lines.",
    "Volunteers from the conservation group planted more than five thousand native trees along the eroding riverbank over the weekend.",
]

from datasets import load_dataset

TEST_TEXTS = []
source = None

# 1. Try LM1B (the real target)
try:
    print("Trying HF lm1b (test, streaming) …")
    stream = load_dataset('lm1b', split='test', streaming=True, trust_remote_code=True)
    for item in stream:
        t = item['text'].strip()
        if len(t) >= 30:
            TEST_TEXTS.append(t)
        if len(TEST_TEXTS) >= max(NUM_SAMPLES * 2, 32):
            break
    if TEST_TEXTS:
        source = 'lm1b/test'
except Exception as e:
    print(f"  lm1b load failed ({type(e).__name__}): falling back to wikitext.")

# 2. Fall back to wikitext-2
if not TEST_TEXTS:
    try:
        print("Trying HF wikitext-2-raw (test) …")
        stream = load_dataset('wikitext', 'wikitext-2-raw-v1', split='test', streaming=True)
        for item in stream:
            t = item['text'].strip()
            if len(t) >= 30 and not t.startswith('='):
                TEST_TEXTS.append(t)
            if len(TEST_TEXTS) >= max(NUM_SAMPLES * 2, 32):
                break
        if TEST_TEXTS:
            source = 'wikitext-2-raw-v1/test'
    except Exception as e:
        print(f"  wikitext load failed ({type(e).__name__}): using hardcoded fallback.")

# 3. Hardcoded fallback (works offline, always)
if not TEST_TEXTS:
    TEST_TEXTS = list(FALLBACK_SENTENCES)
    source = 'fallback_sentences (in-notebook)'

print(f"TEST_TEXTS source: {source}  |  n = {len(TEST_TEXTS)}")
print(f"example: {TEST_TEXTS[0][:100]}")

## 1. Load models

Loads the trained BertDiffused checkpoint plus both GPT-2 baselines. Checkpoint
is expected at `best_model/best_model.pt` — point `CKPT_PATH` elsewhere if yours
lives on Drive or in MLflow artifacts.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from model import BertMoEDiffusion, LogLinearNoiseSchedule

bert_tokenizer = AutoTokenizer.from_pretrained(cfg['model']['backbone'])
noise_schedule = LogLinearNoiseSchedule()

moe_cfg = cfg['model']['moe']

diff_model = None
if Path(CKPT_PATH).exists():
    print(f"Loading BertDiffused from {CKPT_PATH} …")
    ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=False)
    saved_cfg = ckpt.get('config', cfg)
    saved_moe = saved_cfg.get('model', {}).get('moe', moe_cfg)

    diff_model = BertMoEDiffusion(
        bert_model_name=saved_cfg.get('model', {}).get('backbone', cfg['model']['backbone']),
        moe_layers=saved_moe.get('moe_layers', moe_cfg['moe_layers']),
        num_experts=saved_moe.get('num_experts', moe_cfg['num_experts']),
        num_experts_per_token=saved_moe.get('num_experts_per_token', moe_cfg['num_experts_per_token']),
        expert_hidden_multiplier=saved_moe.get('expert_hidden_multiplier', moe_cfg['expert_hidden_multiplier']),
        router_jitter=0.0,
        router_z_loss_coef=saved_moe.get('router_z_loss_coef', moe_cfg['router_z_loss_coef']),
        router_aux_loss_coef=saved_moe.get('router_aux_loss_coef', moe_cfg['router_aux_loss_coef']),
        time_embed_dim=saved_cfg.get('model', {}).get('time_embed_dim', cfg['model']['time_embed_dim']),
        use_time_conditioning=saved_cfg.get('model', {}).get('use_time_conditioning', cfg['model']['use_time_conditioning']),
        dropout=0.0,
        lora_enabled=False,
    )
    state = ckpt.get('model', ckpt)
    missing, unexpected = diff_model.load_state_dict(state, strict=False)
    print(f"  missing keys:    {len(missing)}")
    print(f"  unexpected keys: {len(unexpected)}")
    diff_model.set_mask_token_id(bert_tokenizer.mask_token_id)
    diff_model.to(device).eval()
    ps = diff_model.trainable_parameters_summary()
    print(f"  total params:    {ps['total']:,}")
else:
    print(f"[skip] {CKPT_PATH} not found — BertDiffused will be skipped in this run.")
    print("      Update CKPT_PATH above to point at your trained checkpoint.")

In [ ]:
print("Loading GPT-2 (117M) …")
gpt2_tok = AutoTokenizer.from_pretrained('gpt2')
gpt2_tok.pad_token = gpt2_tok.eos_token
gpt2_mdl = AutoModelForCausalLM.from_pretrained('gpt2').to(device)
gpt2_mdl.eval()
print(f"  params: {sum(p.numel() for p in gpt2_mdl.parameters()):,}")

print("Loading GPT-2 Medium (345M) …")
gpt2m_tok = AutoTokenizer.from_pretrained('gpt2-medium')
gpt2m_tok.pad_token = gpt2m_tok.eos_token
gpt2m_mdl = AutoModelForCausalLM.from_pretrained('gpt2-medium').to(device)
gpt2m_mdl.eval()
print(f"  params: {sum(p.numel() for p in gpt2m_mdl.parameters()):,}")

## 1b. Model Inventory

All three models side-by-side: params, trainable params, checkpoint size. The
**best trained model** (BertDiffused, ours) is loaded from
`best_model/best_model.pt` — the other two are pretrained from HuggingFace and
used zero-shot.

In [ ]:
def _count_params(m):
    tot = sum(p.numel() for p in m.parameters())
    trn = sum(p.numel() for p in m.parameters() if p.requires_grad)
    return tot, trn

inventory_rows = []
if diff_model is not None:
    tot, trn = _count_params(diff_model)
    inventory_rows.append({
        'model': 'BertDiffused (ours) ★',
        'family': 'Masked Diffusion + MoE',
        'total_params_M': tot / 1e6,
        'trainable_params_M': trn / 1e6,
        'source': CKPT_PATH,
    })
tot, trn = _count_params(gpt2_mdl)
inventory_rows.append({
    'model': 'GPT-2 (117M)',
    'family': 'Autoregressive',
    'total_params_M': tot / 1e6,
    'trainable_params_M': trn / 1e6,
    'source': 'openai-community/gpt2 (zero-shot)',
})
tot, trn = _count_params(gpt2m_mdl)
inventory_rows.append({
    'model': 'GPT-2 Medium (345M)',
    'family': 'Autoregressive',
    'total_params_M': tot / 1e6,
    'trainable_params_M': trn / 1e6,
    'source': 'openai-community/gpt2-medium (zero-shot)',
})

df_inv = pd.DataFrame(inventory_rows)
df_inv.to_csv(OUTPUT_DIR / 'model_inventory.csv', index=False)
df_inv.round(1)

## 2. Task 1 — Text Infilling

Masks the middle 30% of an LM1B test sentence and asks each model to fill it.
Diffusion gets bidirectional access to prefix+suffix; GPT-2 sees only the prefix
(standard causal decoding) — this is the architectural gap we want to measure.

In [ ]:
from tasks.infilling import (
    diffusion_infill,
    gpt2_infill,
    compute_bleu,
    compute_generative_ppl,
)

def build_infilling_from_texts(texts, tokenizer, num_examples, seq_len, mask_frac=0.3):
    """Same contract as tasks.infilling.build_infilling_examples, but takes
    a pre-fetched list of strings so it works without HF-dataset streaming."""
    out = []
    for text in texts:
        if len(out) >= num_examples:
            break
        enc = tokenizer(text, max_length=seq_len, truncation=True,
                        padding='max_length', return_tensors='pt')
        ids = enc['input_ids'].squeeze(0)
        real_len = int(enc['attention_mask'].squeeze(0).sum().item())
        if real_len < 20:
            continue
        span_len = max(1, int(real_len * mask_frac))
        margin = (real_len - span_len) // 2
        ps, pe = margin, margin + span_len
        out.append({
            'full_ids':    ids,
            'prefix_ids':  ids[:ps].clone(),
            'suffix_ids':  ids[pe:real_len].clone(),
            'span_ids':    ids[ps:pe].clone(),
            'span_start':  ps,
            'span_end':    pe,
            'real_len':    real_len,
            'full_text':   text,
        })
    return out

print("Building infilling examples from TEST_TEXTS …")
t0 = time.time()
examples = build_infilling_from_texts(
    TEST_TEXTS, bert_tokenizer, num_examples=NUM_SAMPLES, seq_len=SEQ_LEN,
)
print(f"  → {len(examples)} examples in {time.time()-t0:.1f}s  (source={source})")
reference_texts = [ex['full_text'] for ex in examples]

In [ ]:
task1_results = {}

# ── BertDiffused ─────────────────────────────────────────────────────────
if diff_model is not None:
    print(f"BertDiffused infilling (T={NUM_STEPS}) …")
    t0 = time.time()
    diff_texts = diffusion_infill(
        diff_model, examples, noise_schedule, bert_tokenizer, NUM_STEPS, device,
    )
    bleu = compute_bleu(diff_texts, reference_texts)
    gppl = compute_generative_ppl(diff_texts, device=device)
    task1_results['BertDiffused (ours)'] = {'bleu4': bleu, 'generative_ppl': gppl, 'wall_s': time.time()-t0}
    print(f"  BLEU-4: {bleu:.2f}  |  Gen-PPL: {gppl:.2f}  |  {time.time()-t0:.1f}s")

# ── GPT-2 (117M) ─────────────────────────────────────────────────────────
print("GPT-2 (117M) infilling …")
t0 = time.time()
gpt2_texts = gpt2_infill(gpt2_mdl, gpt2_tok, examples, bert_tokenizer, device)
bleu = compute_bleu(gpt2_texts, reference_texts)
gppl = compute_generative_ppl(gpt2_texts, device=device)
task1_results['GPT-2 (117M)'] = {'bleu4': bleu, 'generative_ppl': gppl, 'wall_s': time.time()-t0}
print(f"  BLEU-4: {bleu:.2f}  |  Gen-PPL: {gppl:.2f}  |  {time.time()-t0:.1f}s")

# ── GPT-2 Medium (345M) ──────────────────────────────────────────────────
print("GPT-2 Medium (345M) infilling …")
t0 = time.time()
gpt2m_texts = gpt2_infill(gpt2m_mdl, gpt2m_tok, examples, bert_tokenizer, device)
bleu = compute_bleu(gpt2m_texts, reference_texts)
gppl = compute_generative_ppl(gpt2m_texts, device=device)
task1_results['GPT-2 Medium (345M)'] = {'bleu4': bleu, 'generative_ppl': gppl, 'wall_s': time.time()-t0}
print(f"  BLEU-4: {bleu:.2f}  |  Gen-PPL: {gppl:.2f}  |  {time.time()-t0:.1f}s")

df1 = pd.DataFrame(task1_results).T
df1.to_csv(OUTPUT_DIR / 'task1_infilling.csv')
df1.round(3)

## 3. Task 2 — Keyword-Constrained Generation

Given 3 keywords, each model must produce a 64-token sentence containing all of
them. Diffusion pins them at fixed positions (satisfaction is architectural);
GPT-2 generates with a keyword-inclusion prompt (satisfaction is approximate).

In [ ]:
from tasks.constrained_gen import (
    build_constrained_examples,
    diffusion_constrained_gen,
    gpt2_constrained_gen,
    keyword_satisfaction_rate,
    compute_mauve,
)

print("Building constrained-generation examples …")
kw_examples = build_constrained_examples(bert_tokenizer, seq_len=KW_SEQ_LEN, num_examples=NUM_SAMPLES)
print(f"  → {len(kw_examples)} examples")

# Reference texts for MAUVE — reuse TEST_TEXTS (clipped to KW_SEQ_LEN) rather
# than calling the library's LM1B-backed helper.
ref_texts = []
for text in TEST_TEXTS[:NUM_SAMPLES]:
    enc = bert_tokenizer(text, max_length=KW_SEQ_LEN, truncation=True)
    ref_texts.append(bert_tokenizer.decode(enc['input_ids'], skip_special_tokens=True))
print(f"  → {len(ref_texts)} reference texts for MAUVE")

task2_results = {}

if diff_model is not None:
    print(f"BertDiffused constrained-gen (T={NUM_STEPS}) …")
    t0 = time.time()
    diff_texts_kw = diffusion_constrained_gen(
        diff_model, kw_examples, noise_schedule, bert_tokenizer, NUM_STEPS, device,
    )
    kw_rate, _ = keyword_satisfaction_rate(diff_texts_kw, kw_examples)
    mauve = compute_mauve(diff_texts_kw, ref_texts, max_text_length=KW_SEQ_LEN)
    gppl = compute_generative_ppl(diff_texts_kw, device=device)
    task2_results['BertDiffused (ours)'] = {
        'keyword_satisfaction_rate': kw_rate, 'mauve': mauve, 'generative_ppl': gppl,
        'wall_s': time.time()-t0,
    }
    print(f"  KW-Sat: {kw_rate:.2%}  |  MAUVE: {mauve:.4f}  |  Gen-PPL: {gppl:.2f}  |  {time.time()-t0:.1f}s")

print("GPT-2 (117M) constrained-gen …")
t0 = time.time()
gpt2_texts_kw = gpt2_constrained_gen(gpt2_mdl, gpt2_tok, kw_examples, device, max_length=KW_SEQ_LEN)
kw_rate, _ = keyword_satisfaction_rate(gpt2_texts_kw, kw_examples)
mauve = compute_mauve(gpt2_texts_kw, ref_texts, max_text_length=KW_SEQ_LEN)
gppl = compute_generative_ppl(gpt2_texts_kw, device=device)
task2_results['GPT-2 (117M)'] = {
    'keyword_satisfaction_rate': kw_rate, 'mauve': mauve, 'generative_ppl': gppl,
    'wall_s': time.time()-t0,
}
print(f"  KW-Sat: {kw_rate:.2%}  |  MAUVE: {mauve:.4f}  |  Gen-PPL: {gppl:.2f}  |  {time.time()-t0:.1f}s")

print("GPT-2 Medium (345M) constrained-gen …")
t0 = time.time()
gpt2m_texts_kw = gpt2_constrained_gen(gpt2m_mdl, gpt2m_tok, kw_examples, device, max_length=KW_SEQ_LEN)
kw_rate, _ = keyword_satisfaction_rate(gpt2m_texts_kw, kw_examples)
mauve = compute_mauve(gpt2m_texts_kw, ref_texts, max_text_length=KW_SEQ_LEN)
gppl = compute_generative_ppl(gpt2m_texts_kw, device=device)
task2_results['GPT-2 Medium (345M)'] = {
    'keyword_satisfaction_rate': kw_rate, 'mauve': mauve, 'generative_ppl': gppl,
    'wall_s': time.time()-t0,
}
print(f"  KW-Sat: {kw_rate:.2%}  |  MAUVE: {mauve:.4f}  |  Gen-PPL: {gppl:.2f}  |  {time.time()-t0:.1f}s")

df2 = pd.DataFrame(task2_results).T
df2.to_csv(OUTPUT_DIR / 'task2_constrained.csv')
df2.round(3)

### 3b. Qualitative samples — same prompts across all three models

Shows the first few outputs per model side-by-side so you can eyeball whether the
best trained model actually produces coherent text (not just numerically good).
Metrics can be gamed; humans reading generations can't.

In [ ]:
N_QUAL = min(3, NUM_SAMPLES)

# ── Task 1 side-by-side ──────────────────────────────────────────────────
print('=' * 80)
print('TASK 1 · Text Infilling · first', N_QUAL, 'examples')
print('=' * 80)
for i in range(N_QUAL):
    print(f'\n[{i+1}] Ground truth: {examples[i]["full_text"][:140]}')
    if diff_model is not None:
        print(f'    BertDiffused ★ : {diff_texts[i][:140]}')
    print(f'    GPT-2 (117M)    : {gpt2_texts[i][:140]}')
    print(f'    GPT-2 Medium    : {gpt2m_texts[i][:140]}')

# ── Task 2 side-by-side ──────────────────────────────────────────────────
print('\n' + '=' * 80)
print('TASK 2 · Keyword-Constrained Generation · first', N_QUAL, 'examples')
print('=' * 80)
for i in range(N_QUAL):
    kws = ', '.join(kw_examples[i]['keywords'])
    print(f'\n[{i+1}] Keywords: [{kws}]')
    if diff_model is not None:
        hit = all(k.lower() in diff_texts_kw[i].lower() for k in kw_examples[i]['keywords'])
        mark = 'OK' if hit else 'MISS'
        print(f'    BertDiffused ★ [{mark}]: {diff_texts_kw[i][:140]}')
    hit = all(k.lower() in gpt2_texts_kw[i].lower() for k in kw_examples[i]['keywords'])
    mark = 'OK' if hit else 'MISS'
    print(f'    GPT-2 (117M)    [{mark}]: {gpt2_texts_kw[i][:140]}')
    hit = all(k.lower() in gpt2m_texts_kw[i].lower() for k in kw_examples[i]['keywords'])
    mark = 'OK' if hit else 'MISS'
    print(f'    GPT-2 Medium    [{mark}]: {gpt2m_texts_kw[i][:140]}')

# Persist a small qualitative-samples file for reference
qual_samples = []
for i in range(N_QUAL):
    row = {
        'task': 'infilling',
        'reference': examples[i]['full_text'],
        'gpt2_117M': gpt2_texts[i],
        'gpt2_medium_345M': gpt2m_texts[i],
    }
    if diff_model is not None:
        row['bertdiffused'] = diff_texts[i]
    qual_samples.append(row)
for i in range(N_QUAL):
    row = {
        'task': 'constrained',
        'keywords': ', '.join(kw_examples[i]['keywords']),
        'gpt2_117M': gpt2_texts_kw[i],
        'gpt2_medium_345M': gpt2m_texts_kw[i],
    }
    if diff_model is not None:
        row['bertdiffused'] = diff_texts_kw[i]
    qual_samples.append(row)
pd.DataFrame(qual_samples).to_csv(OUTPUT_DIR / 'qualitative_samples.csv', index=False)

## 4. Task 3 — Unconditional Generation Quality vs. Steps

Generates text from scratch for each model and measures Gen-PPL under the same
GPT-2 scorer. For diffusion, sweeps the number of reverse steps `T` to expose
the quality/speed trade-off; AR models are plotted as a horizontal line since
they have no `T` knob.

In [ ]:
unconditional = {}

# ── GPT-2 baselines (constant w.r.t. T) ──────────────────────────────────
for mdl, tok, name in [
    (gpt2_mdl,  gpt2_tok,  'GPT-2 (117M)'),
    (gpt2m_mdl, gpt2m_tok, 'GPT-2 Medium (345M)'),
]:
    print(f"{name} unconditional …")
    texts = []
    with torch.no_grad():
        for _ in range(0, min(NUM_SAMPLES, 64), 16):
            n = min(16, min(NUM_SAMPLES, 64) - len(texts))
            prompt_ids = torch.tensor([[tok.bos_token_id]] * n, device=device)
            out = mdl.generate(
                prompt_ids, max_length=SEQ_LEN, do_sample=True,
                temperature=1.0, top_p=0.9, pad_token_id=tok.eos_token_id,
            )
            for row in out:
                texts.append(tok.decode(row.tolist(), skip_special_tokens=True))
    gppl = compute_generative_ppl(texts, device=device)
    unconditional[name] = [(None, gppl)]
    print(f"  Gen-PPL: {gppl:.2f}")

# ── BertDiffused sweep over T ────────────────────────────────────────────
if diff_model is not None:
    sweep = []
    for T in STEPS_SWEEP:
        print(f"BertDiffused unconditional, T={T} …")
        t0 = time.time()
        gen_ids = diff_model.sample(
            batch_size=min(NUM_SAMPLES, 64), seq_len=SEQ_LEN,
            num_steps=T, noise_schedule=noise_schedule,
            tokenizer=bert_tokenizer, device=device,
        )
        texts = [bert_tokenizer.decode(r.tolist(), skip_special_tokens=True) for r in gen_ids]
        gppl = compute_generative_ppl(texts, device=device)
        sweep.append((T, gppl))
        print(f"  T={T:4d} → Gen-PPL: {gppl:.2f}  |  {time.time()-t0:.1f}s")
    unconditional['BertDiffused (ours)'] = sweep

# Flatten to CSV
rows = []
for name, entries in unconditional.items():
    for T, ppl in entries:
        rows.append({'model': name, 'diffusion_steps': T if T is not None else 'N/A', 'generative_ppl': ppl})
df3 = pd.DataFrame(rows)
df3.to_csv(OUTPUT_DIR / 'task3_unconditional.csv', index=False)
df3.round(3)

## 5. Test-Set Likelihood (bits/token)

A direct, scorer-independent comparison:

- **Diffusion NELBO per token** (`eval.bpd.diffusion_nelbo_per_token`) — Monte-Carlo
  estimate averaged over several t values, scored on the same LM1B test batch.
- **AR cross-entropy per token** (`eval.bpd.ar_cross_entropy_per_token`) — standard
  causal-LM CE over the same texts.

Both are converted to **bits/token** so they share an axis. This is a noisier
comparison than BLEU/MAUVE (the MDLM objective is an ELBO, not exact PPL), but
it's the closest single scalar for ranking models on held-out data.

In [ ]:
from eval.bpd import diffusion_nelbo_per_token, ar_cross_entropy_per_token, nats_to_bits

# Pull a small deterministic batch from TEST_TEXTS (no HF streaming needed)
raw_texts, tokenized_ids, tokenized_mask = [], [], []
for text in TEST_TEXTS:
    if len(raw_texts) >= BPD_BATCHES * BPD_BATCH_SIZE:
        break
    text = text.strip()
    if len(text) < 30:
        continue
    enc = bert_tokenizer(
        text, max_length=SEQ_LEN, truncation=True,
        padding='max_length', return_tensors='pt',
    )
    raw_texts.append(text)
    tokenized_ids.append(enc['input_ids'].squeeze(0))
    tokenized_mask.append(enc['attention_mask'].squeeze(0))

print(f"likelihood eval set: {len(raw_texts)} examples  (source={source})")

bpd_results = {}

if diff_model is not None:
    print("BertDiffused NELBO …")
    nelbos = []
    for b in range(0, len(raw_texts), BPD_BATCH_SIZE):
        ids = torch.stack(tokenized_ids[b:b+BPD_BATCH_SIZE])
        msk = torch.stack(tokenized_mask[b:b+BPD_BATCH_SIZE])
        n = diffusion_nelbo_per_token(
            diff_model, ids, msk, noise_schedule,
            mask_token_id=bert_tokenizer.mask_token_id,
            num_t_samples=BPD_NUM_T, device=device,
        )
        nelbos.append(n)
    nats = float(np.mean(nelbos))
    bpd_results['BertDiffused (ours)'] = {'nats_per_token': nats, 'bits_per_token': nats_to_bits(nats)}
    print(f"  {nats:.3f} nats/tok  |  {nats_to_bits(nats):.3f} bits/tok")

print("GPT-2 (117M) CE …")
nats = ar_cross_entropy_per_token(gpt2_mdl, gpt2_tok, raw_texts, max_length=SEQ_LEN, batch_size=BPD_BATCH_SIZE, device=device)
bpd_results['GPT-2 (117M)'] = {'nats_per_token': nats, 'bits_per_token': nats_to_bits(nats)}
print(f"  {nats:.3f} nats/tok  |  {nats_to_bits(nats):.3f} bits/tok")

print("GPT-2 Medium (345M) CE …")
nats = ar_cross_entropy_per_token(gpt2m_mdl, gpt2m_tok, raw_texts, max_length=SEQ_LEN, batch_size=BPD_BATCH_SIZE, device=device)
bpd_results['GPT-2 Medium (345M)'] = {'nats_per_token': nats, 'bits_per_token': nats_to_bits(nats)}
print(f"  {nats:.3f} nats/tok  |  {nats_to_bits(nats):.3f} bits/tok")

df4 = pd.DataFrame(bpd_results).T
df4.to_csv(OUTPUT_DIR / 'task4_bpd.csv')
df4.round(3)

## 6. Plots

In [ ]:
# ── Task 1: Infilling — BLEU-4 & Gen-PPL ─────────────────────────────────
if task1_results:
    models = list(task1_results.keys())
    bleus  = [task1_results[m]['bleu4'] for m in models]
    ppls   = [task1_results[m]['generative_ppl'] for m in models]
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
    a1.bar(models, bleus, color=['#2196F3', '#FF5722', '#4CAF50'][:len(models)])
    a1.set_ylabel('BLEU-4 (↑)'); a1.set_title('Task 1 · Infilling · BLEU-4')
    a1.tick_params(axis='x', rotation=15); a1.grid(axis='y', alpha=0.3)
    a2.bar(models, ppls, color=['#2196F3', '#FF5722', '#4CAF50'][:len(models)])
    a2.set_ylabel('Gen-PPL (↓)'); a2.set_title('Task 1 · Infilling · Gen-PPL')
    a2.tick_params(axis='x', rotation=15); a2.grid(axis='y', alpha=0.3)
    plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'task1_plot.png', dpi=150); plt.show()

# ── Task 2: KW-Sat, MAUVE, Gen-PPL ───────────────────────────────────────
if task2_results:
    models = list(task2_results.keys())
    kw = [task2_results[m]['keyword_satisfaction_rate']*100 for m in models]
    mv = [task2_results[m]['mauve'] for m in models]
    pp = [task2_results[m]['generative_ppl'] for m in models]
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, vals, ylabel, title in zip(
        axes, [kw, mv, pp],
        ['KW-Sat (%)', 'MAUVE (↑)', 'Gen-PPL (↓)'],
        ['Task 2 · KW-Sat', 'Task 2 · MAUVE', 'Task 2 · Gen-PPL'],
    ):
        ax.bar(models, vals, color=['#2196F3', '#FF5722', '#4CAF50'][:len(models)])
        ax.set_ylabel(ylabel); ax.set_title(title)
        ax.tick_params(axis='x', rotation=15); ax.grid(axis='y', alpha=0.3)
    plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'task2_plot.png', dpi=150); plt.show()

# ── Task 3: unconditional PPL vs T ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4.5))
for name, entries in unconditional.items():
    if entries[0][0] is None:
        ax.axhline(entries[0][1], linestyle='--', label=name)
    else:
        xs = [e[0] for e in entries]; ys = [e[1] for e in entries]
        ax.plot(xs, ys, marker='o', label=name)
ax.set_xscale('log'); ax.set_xlabel('Diffusion steps T (log)'); ax.set_ylabel('Gen-PPL (↓)')
ax.set_title('Task 3 · Unconditional Generation · Gen-PPL vs T')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'task3_plot.png', dpi=150); plt.show()

# ── Task 4: bits/token ────────────────────────────────────────────────────
if bpd_results:
    models = list(bpd_results.keys())
    bits   = [bpd_results[m]['bits_per_token'] for m in models]
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(models, bits, color=['#2196F3', '#FF5722', '#4CAF50'][:len(models)])
    ax.set_ylabel('bits / token (↓)'); ax.set_title('Task 4 · Test-set likelihood (lower = better)')
    ax.tick_params(axis='x', rotation=15); ax.grid(axis='y', alpha=0.3)
    plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'task4_plot.png', dpi=150); plt.show()

### 6b. Latency & throughput

Wall-clock per model on the Task 1 (infilling) and Task 2 (constrained) workloads
— captured during the runs above. Diffusion pays for its bidirectional context
with T forward passes; AR pays one forward per token. Don't read absolute
numbers — read the *ratio* between models.

In [ ]:
latency_rows = []
all_models = []
if diff_model is not None:
    all_models.append('BertDiffused (ours)')
all_models += ['GPT-2 (117M)', 'GPT-2 Medium (345M)']

for m in all_models:
    t1 = task1_results.get(m, {}).get('wall_s', float('nan'))
    t2 = task2_results.get(m, {}).get('wall_s', float('nan'))
    latency_rows.append({
        'model': m,
        'task1_infilling_s': t1,
        'task1_samples_per_s': (NUM_SAMPLES / t1) if t1 == t1 and t1 > 0 else float('nan'),
        'task2_constrained_s': t2,
        'task2_samples_per_s': (NUM_SAMPLES / t2) if t2 == t2 and t2 > 0 else float('nan'),
    })

df_lat = pd.DataFrame(latency_rows)
df_lat.to_csv(OUTPUT_DIR / 'latency.csv', index=False)

# Bar chart: samples/s per model (higher = faster)
fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(df_lat))
w = 0.35
ax.bar(x - w/2, df_lat['task1_samples_per_s'], w, label='Task 1 infilling', color='#2196F3')
ax.bar(x + w/2, df_lat['task2_samples_per_s'], w, label='Task 2 constrained', color='#FF5722')
ax.set_xticks(x); ax.set_xticklabels(df_lat['model'], rotation=15)
ax.set_ylabel('samples / s (↑ faster)')
ax.set_title('Throughput by Model')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'latency_plot.png', dpi=150); plt.show()

df_lat.round(2)

## 7. Consolidated Summary

In [ ]:
all_results = {
    'task1_infilling': task1_results,
    'task2_constrained': task2_results,
    'task3_unconditional': {
        m: [(str(T), p) for T, p in entries]
        for m, entries in unconditional.items()
    },
    'task4_bpd': bpd_results,
    'config': {
        'num_samples': NUM_SAMPLES, 'num_steps': NUM_STEPS,
        'steps_sweep': STEPS_SWEEP, 'seq_len': SEQ_LEN, 'kw_seq_len': KW_SEQ_LEN,
        'ckpt_path': CKPT_PATH, 'device': str(device),
    },
}
with open(OUTPUT_DIR / 'all_results.json', 'w') as f:
    json.dump(all_results, f, indent=2)

print('=' * 80)
print('BASELINE COMPARISON · SUMMARY')
print('=' * 80)
print('\nTask 1 · Text Infilling')
print(df1.round(3))
print('\nTask 2 · Keyword-Constrained Generation')
print(df2.round(3))
print('\nTask 3 · Unconditional Generation')
print(df3.round(3))
print('\nTask 4 · Test-set likelihood')
print(df4.round(3))
print(f'\nFull JSON: {OUTPUT_DIR / "all_results.json"}')

### 7b. Unified comprehensive comparison

All metrics across all tasks in one wide table; the best value per column is
marked with ★. For diffusion, the Task-3 Gen-PPL uses the best T from the sweep.

In [ ]:
def _best_T_ppl(entries):
    """From a sweep of (T, ppl) tuples return the lowest ppl."""
    if entries is None:
        return float('nan')
    ppls = [p for _, p in entries if p is not None]
    return min(ppls) if ppls else float('nan')

rows = []
for m in all_models:
    t1 = task1_results.get(m, {})
    t2 = task2_results.get(m, {})
    uc = unconditional.get(m, None)
    bp = bpd_results.get(m, {})
    rows.append({
        'model': m,
        'T1_BLEU4':        t1.get('bleu4', float('nan')),
        'T1_GenPPL':       t1.get('generative_ppl', float('nan')),
        'T2_KWSat_pct':    100 * t2.get('keyword_satisfaction_rate', float('nan')),
        'T2_MAUVE':        t2.get('mauve', float('nan')),
        'T2_GenPPL':       t2.get('generative_ppl', float('nan')),
        'T3_best_GenPPL':  _best_T_ppl(uc),
        'T4_bits_per_tok': bp.get('bits_per_token', float('nan')),
    })
df_all = pd.DataFrame(rows).set_index('model')

# Winner per column (↑ for higher-is-better, ↓ for lower-is-better)
higher_is_better = {'T1_BLEU4', 'T2_KWSat_pct', 'T2_MAUVE'}
winners = {}
for col in df_all.columns:
    vals = df_all[col].dropna()
    if vals.empty:
        continue
    winners[col] = vals.idxmax() if col in higher_is_better else vals.idxmin()

def _mark(val, col, idx):
    if np.isnan(val):
        return 'n/a'
    return f"{val:.2f} ★" if winners.get(col) == idx else f"{val:.2f}"

marked = df_all.copy().astype(object)
for col in df_all.columns:
    for idx in df_all.index:
        marked.loc[idx, col] = _mark(df_all.loc[idx, col], col, idx)

marked.to_csv(OUTPUT_DIR / 'unified_comparison.csv')
print('=' * 100)
print('UNIFIED COMPREHENSIVE COMPARISON   (★ = best in column)')
print('=' * 100)
print(marked.to_string())
print()
print('Column guide:')
print('  T1_BLEU4        — Task 1 infilling, BLEU-4 (↑)')
print('  T1_GenPPL       — Task 1 infilling, Gen-PPL (↓)')
print('  T2_KWSat_pct    — Task 2 keyword satisfaction % (↑)')
print('  T2_MAUVE        — Task 2 MAUVE (↑)')
print('  T2_GenPPL       — Task 2 Gen-PPL (↓)')
print('  T3_best_GenPPL  — Task 3 best Gen-PPL over the T sweep (↓)')
print('  T4_bits_per_tok — Task 4 test-set likelihood in bits/token (↓)')

---

### What the numbers mean

- **BLEU-4** (infilling) — n-gram overlap of the reconstructed sentence with the
  ground-truth. Diffusion should dominate because it actually *sees* the suffix.
- **Gen-PPL** — fluency under a held-out GPT-2 scorer. Mildly favours AR-style
  outputs. MAUVE is the more paradigm-agnostic quality metric.
- **KW-Sat** — fraction of outputs that contain *all* required keywords. For
  diffusion this should be ≈100% (keywords are pinned); GPT-2's number depends
  on how well it follows the prompt.
- **bits/token** (Task 4) — lower is better. Diffusion uses an ELBO, GPT-2 uses
  exact CE; the gap between the two is expected and not a bug.

### Reproducing at scale

On a GPU host, bump `NUM_SAMPLES` to 500 and `NUM_STEPS` to 1000 to match the
paper-style run. Outputs land in `results/baselines/`.